Choose your results folder and load all its method CSV files without changing their data.


In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "JPNotebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent


# Choose the folder containing your method CSVs; every CSV in this folder will be loaded.
RESULTS_FOLDER = PROJECT_ROOT / "Results" / "thu_oct_8"
CSV_FILES = sorted(path.name for path in RESULTS_FOLDER.glob("*.csv"))

if not CSV_FILES:
    raise FileNotFoundError(f"No CSV files found in {RESULTS_FOLDER}")

csv_data = {}

for filename in CSV_FILES:
    csv_data[filename] = pd.read_csv(RESULTS_FOLDER / filename)

print("Loaded CSVs:", ", ".join(CSV_FILES))


Prepare the loaded scores, average each client’s accuracy across rounds, and save the graph and PDF under CellResults.


In [ ]:
import sys
import matplotlib.pyplot as plt

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from analysis.results import client_accuracies, training_losses


# Check the configured clients and rounds in each CSV before averaging their accuracy scores.
client_data = client_accuracies(csv_data)

average_accuracies = (
    client_data.groupby(["client_id", "method"])["client_accuracy"].mean().unstack()
)


# Keep the methods in this order so running or saving them in a different order does not change the graph.
method_names = {
    "fedavg": "FedAvg",
    "fairrfl": "FairRFL",
    "fairmedian": "FairMedian",
    "fedcda": "FedCDA",
    "fedmut": "FedMut",
}
methods = [method for method in method_names if method in average_accuracies.columns]
methods += sorted(set(average_accuracies.columns) - set(method_names))
average_accuracies = average_accuracies[methods]


# Put methods on the x-axis and draw one line per client, with average accuracy shown as a percentage.
plot_data = average_accuracies.T * 100
plot_data.index = [method_names.get(method, method) for method in methods]
plot_data.columns = [f"Client {client_id}" for client_id in plot_data.columns]

ax = plot_data.plot(marker="o", figsize=(12, 6), colormap="turbo")
ax.set_title("Average client accuracy across rounds")
ax.set_xlabel("Method")
ax.set_ylabel("Average accuracy (%)")
ax.set_ylim(0, 100)
ax.grid(axis="y", alpha=0.3)
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", ncol=2, fontsize=8)

fig = ax.figure
fig.tight_layout()


# Use the CSV folder's name inside CellResults and keep each analysis run in its own numbered folder.
comparison_folder = PROJECT_ROOT / "JPNotebooks" / "CellResults" / RESULTS_FOLDER.name
version = 1
result_folder = comparison_folder / f"result_{version}"

while result_folder.exists():
    version += 1
    result_folder = comparison_folder / f"result_{version}"

result_folder.mkdir(parents=True)
pdf_path = result_folder / "average_client_accuracy_by_method.pdf"
fig.savefig(pdf_path, bbox_inches="tight")
plt.show()
plt.close(fig)

print("Saved PDF:", pdf_path)
(average_accuracies * 100).round(2)


Compare each method’s average local training loss across rounds and save its PDF beside the accuracy graph.


In [ ]:
# Check the selected clients, local epochs, batch numbers and sample counts before averaging the logged losses.
loss_data = training_losses(csv_data)


# Average across clients and local epochs within each round, weighting each batch by its actual sample count.
round_totals = loss_data.groupby(["round_number", "method"])[["weighted_loss", "samples"]].sum()
average_losses = (round_totals["weighted_loss"] / round_totals["samples"]).unstack("method")
average_losses = average_losses[[method for method in methods if method in average_losses.columns]]


# Draw one line per method; these are local training losses, rather than losses measured on the global test set.
loss_plot = average_losses.rename(columns=method_names)
loss_ax = loss_plot.plot(marker="o", figsize=(12, 6))
loss_ax.set_title("Average local training loss per round")
loss_ax.set_xlabel("Communication round")
loss_ax.set_ylabel("Cross-entropy loss")
loss_ax.set_xticks(average_losses.index)
loss_ax.set_ylim(bottom=0)
loss_ax.grid(alpha=0.3)
loss_ax.legend(title="Method")

loss_fig = loss_ax.figure
loss_fig.tight_layout()


# Save the loss PDF beside the accuracy PDF in the folder created by the previous cell.
loss_pdf_path = result_folder / "average_training_loss_by_round.pdf"
loss_fig.savefig(loss_pdf_path, bbox_inches="tight")
plt.show()
plt.close(loss_fig)

print("Saved PDF:", loss_pdf_path)
average_losses.round(4)


Average each client’s training loss across recorded rounds and compare methods using the same layout as the accuracy graph.


In [ ]:


# Use the batch losses prepared in cell 3 to calculate each client's sample-weighted loss for every recorded round.
client_round_totals = loss_data.groupby(["client_id", "method", "round_number"])[
    ["weighted_loss", "samples"]
].sum()
client_round_losses = client_round_totals["weighted_loss"] / client_round_totals["samples"]


# Average those round losses separately for each client and method, giving every recorded round equal weight.
average_client_losses = (
    client_round_losses.groupby(level=["client_id", "method"]).mean().unstack("method")
)
average_client_losses = average_client_losses[
    [method for method in methods if method in average_client_losses.columns]
]


# Match the accuracy graph's layout: methods on the x-axis and one line per client, with loss shown in its original units.
client_loss_plot = average_client_losses.T
client_loss_plot.index = [method_names.get(method, method) for method in client_loss_plot.index]
client_loss_plot.columns = [f"Client {client_id}" for client_id in client_loss_plot.columns]

client_loss_ax = client_loss_plot.plot(marker="o", figsize=(12, 6), colormap="turbo")
client_loss_ax.set_title("Average client training loss across rounds")
client_loss_ax.set_xlabel("Method")
client_loss_ax.set_ylabel("Average cross-entropy loss")
client_loss_ax.set_ylim(bottom=0)
client_loss_ax.grid(axis="y", alpha=0.3)
client_loss_ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", ncol=2, fontsize=8)

client_loss_fig = client_loss_ax.figure
client_loss_fig.tight_layout()


# Save this client-wise loss graph in the same numbered folder as the other figures from this notebook run.
client_loss_pdf_path = result_folder / "average_client_training_loss_by_method.pdf"
client_loss_fig.savefig(client_loss_pdf_path, bbox_inches="tight")
plt.show()
plt.close(client_loss_fig)

print("Saved PDF:", client_loss_pdf_path)
average_client_losses.round(4)
